# Normal Drift Baseline

Reproduces the drift detection experiment from:
> Kuppa & Le-Khac (2022) *Learn to adapt: Robust drift detection in security domain.*

**Dataset**: CICIDS2017 (Engelen et al. corrected)  
**Protocol**: Drop novel attack classes from training; evaluate drift detection on test set.  
**Threshold**: F1-optimal sweep on calibration/validation split, evaluated on a disjoint held-out set.

| Label | Meaning |
|---|---|
| 1 (positive) | novel/drifted sample (from a dropped class) |
| 0 (negative) | known-class sample |

## 1. Imports & Configuration

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('../..'))

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import DataLoader, TensorDataset

from src.detectors.contrastive_ncm import (
    ContrastiveNCMDetector,
    DriftAutoencoder,
    NCMClassifier,
    train_plain_autoencoder,
)

DATA_DIR        = os.path.abspath('../../data/CICIDS2017_Engelen')
EPOCHS          = 50
LR              = 0.0001
BATCH_SIZE      = 256
TEST_SIZE       = 0.25
DROPPED_CLASSES = ['DoS', 'Patator']
RANDOM_SEED     = 42
DEVICE          = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 2. Load & Preprocess

In [ ]:
DAY_FILES = [
    'Monday-WorkingHours.csv',
    'Tuesday-WorkingHours.csv',
    'Wednesday-WorkingHours.csv',
    'Thursday-WorkingHours.csv',
    'Friday-WorkingHours.csv',
]

print('Loading data...')
df = pd.concat(
    [pd.read_csv(os.path.join(DATA_DIR, f)) for f in DAY_FILES],
    ignore_index=True,
)
print(f'  Total rows: {len(df):,}')

df.drop(columns=['Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Timestamp'],
        inplace=True, errors='ignore')
df['Label'] = df['Label'].apply(lambda x: 'BENIGN' if x.endswith('- Attempted') else x)
df['Label'] = df['Label'].replace({
    'DoS Hulk': 'DoS', 'DoS GoldenEye': 'DoS',
    'DoS slowloris': 'DoS', 'DoS Slowhttptest': 'DoS',
    'Web Attack - Brute Force': 'Web Attack', 'Web Attack - XSS': 'Web Attack',
    'Web Attack - Sql Injection': 'Web Attack',
    'FTP-Patator': 'Patator', 'SSH-Patator': 'Patator',
})

X_raw = df[[c for c in df.columns if c != 'Label']].copy()
y_str = df['Label'].values
X_raw.replace([np.inf, -np.inf], np.nan, inplace=True)
X_raw.dropna(axis=1, how='all', inplace=True)
X_raw.fillna(X_raw.mean(), inplace=True)

X       = StandardScaler().fit_transform(X_raw).astype(np.float32)
le_full = LabelEncoder()
y_all   = le_full.fit_transform(y_str)
input_dim = X.shape[1]
print(f'Features: {input_dim}  |  Classes: {list(le_full.classes_)}')

## 3. Splits

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y_all, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y_all
)

dropped_ids = np.array([le_full.transform([c])[0] for c in DROPPED_CLASSES])
train_mask  = ~np.isin(y_tr, dropped_ids)

le_train    = LabelEncoder()
y_tr_reenc  = le_train.fit_transform(y_tr[train_mask]).astype(np.int64)
num_train_classes = len(le_train.classes_)

# 10 % calibration split — never used for NCM prototype fitting
X_tr_fit, X_cal, y_tr_fit, _ = train_test_split(
    X_tr[train_mask], y_tr_reenc, test_size=0.10,
    random_state=RANDOM_SEED, stratify=y_tr_reenc,
)

# Balanced test set (equal novel/known samples)
rng        = np.random.default_rng(RANDOM_SEED)
te_pos_idx = np.where(np.isin(y_te, dropped_ids))[0]
te_neg_idx = np.where(~np.isin(y_te, dropped_ids))[0]
te_bal_idx = np.concatenate([
    te_pos_idx,
    rng.choice(te_neg_idx, size=len(te_pos_idx), replace=False),
])
X_te_bal    = X_te[te_bal_idx]
y_te_binary = np.isin(y_te[te_bal_idx], dropped_ids).astype(int)

# 50/50 val / held-out split — val selects threshold, held-out evaluates it
val_idx, held_idx = train_test_split(
    np.arange(len(te_bal_idx)), test_size=0.5,
    random_state=RANDOM_SEED, stratify=y_te_binary,
)
X_val_t    = torch.from_numpy(X_te_bal[val_idx])
y_val_bin  = y_te_binary[val_idx]
X_held_t   = torch.from_numpy(X_te_bal[held_idx])
y_held_bin = y_te_binary[held_idx]
X_cal_t    = torch.from_numpy(X_cal)

def make_loader(X_arr, y_arr, shuffle=True):
    return DataLoader(
        TensorDataset(torch.from_numpy(X_arr), torch.from_numpy(y_arr)),
        batch_size=BATCH_SIZE, shuffle=shuffle,
    )

train_loader = make_loader(X_tr_fit, y_tr_fit)

print(f'Dropped classes    : {DROPPED_CLASSES}')
print(f'Train classes ({num_train_classes})  : {[le_full.classes_[i] for i in le_train.classes_]}')
print(f'Train (fit)        : {len(X_tr_fit):,}  |  Cal: {len(X_cal):,}')
print(f'Val                : {len(val_idx):,}  |  Held-out: {len(held_idx):,}')

## 4. Train ContrastiveNCM Detector

In [ ]:
detector = ContrastiveNCMDetector(
    input_dim=input_dim,
    hidden_dim=64,
    latent_dim=32,
    concept_threshold=3.5,
    device=DEVICE,
)
detector.fit(train_loader, epochs=EPOCHS, lr=LR, num_classes=num_train_classes)

## 5. Train Plain AE + NCM Baseline

In [ ]:
plain_ae = DriftAutoencoder(input_dim=input_dim, hidden_dim=64, latent_dim=32).to(DEVICE)
train_plain_autoencoder(plain_ae, train_loader, epochs=EPOCHS, lr=LR)

plain_ae.eval()
all_h, all_y = [], []
with torch.no_grad():
    for x_b, y_b in train_loader:
        all_h.append(plain_ae.encode(x_b.to(DEVICE)))
        all_y.append(y_b)

ncm_plain = NCMClassifier(lambda_1=0.1)
ncm_plain.fit(torch.cat(all_h), torch.cat(all_y), num_train_classes)

## 6. Val-Optimal Threshold Sweep

Sweep quantiles of the calibration-set distance distribution. For each quantile the corresponding distance value is used as the threshold; F1 is evaluated on the validation split. The quantile that maximises val-F1 sets the final threshold, which is then evaluated once on the disjoint held-out set.

In [ ]:
def best_threshold(cal_dists, val_dists, val_labels, n_steps=300):
    best_f1, best_t, best_q = 0.0, 0.0, 0.0
    for q in np.linspace(0.01, 0.99, n_steps):
        t     = float(torch.quantile(cal_dists, q).item())
        preds = (val_dists > t).numpy().astype(int)
        f1    = f1_score(val_labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t, best_q = f1, t, q
    return best_t, best_q, best_f1

# ContrastiveNCM
_, _, cal_dists_c = detector.detect(X_cal_t)
_, _, val_dists_c = detector.detect(X_val_t)
_, _, te_dists_c  = detector.detect(X_held_t)
thresh_c, q_c, val_f1_c = best_threshold(cal_dists_c, val_dists_c, y_val_bin)
print(f'ContrastiveNCM  val-optimal threshold: {thresh_c:.4f}  (q≈{q_c:.2f})  val-F1={val_f1_c:.3f}')

# Plain AE + NCM
with torch.no_grad():
    cal_dists_p = ncm_plain._compute_distance(plain_ae.encode(X_cal_t.to(DEVICE))).min(dim=1).values.cpu()
    val_dists_p = ncm_plain._compute_distance(plain_ae.encode(X_val_t.to(DEVICE))).min(dim=1).values.cpu()
    te_dists_p  = ncm_plain._compute_distance(plain_ae.encode(X_held_t.to(DEVICE))).min(dim=1).values.cpu()
thresh_p, q_p, val_f1_p = best_threshold(cal_dists_p, val_dists_p, y_val_bin)
print(f'Plain AE NCM    val-optimal threshold: {thresh_p:.4f}  (q≈{q_p:.2f})  val-F1={val_f1_p:.3f}')

## 7. Held-Out Evaluation & Diagnostics

In [ ]:
# Held-out metrics
y_pred_c = (te_dists_c > thresh_c).numpy().astype(int)
p_c = precision_score(y_held_bin, y_pred_c, zero_division=0)
r_c = recall_score(y_held_bin, y_pred_c, zero_division=0)
f_c = f1_score(y_held_bin, y_pred_c, zero_division=0)

y_pred_p = (te_dists_p > thresh_p).numpy().astype(int)
p_p = precision_score(y_held_bin, y_pred_p, zero_division=0)
r_p = recall_score(y_held_bin, y_pred_p, zero_division=0)
f_p = f1_score(y_held_bin, y_pred_p, zero_division=0)

# Distance distributions
print('--- Distance distributions (held-out, ContrastiveNCM) ---')
d_novel = te_dists_c[y_held_bin == 1].numpy()
d_known = te_dists_c[y_held_bin == 0].numpy()
print(f"  {'':8s}  {'mean':>7}  {'p25':>7}  {'p50':>7}  {'p75':>7}  {'p95':>7}")
print(f"  {'Known':8s}  {d_known.mean():7.4f}  {np.percentile(d_known,25):7.4f}  "
      f"{np.percentile(d_known,50):7.4f}  {np.percentile(d_known,75):7.4f}  "
      f"{np.percentile(d_known,95):7.4f}")
print(f"  {'Novel':8s}  {d_novel.mean():7.4f}  {np.percentile(d_novel,25):7.4f}  "
      f"{np.percentile(d_novel,50):7.4f}  {np.percentile(d_novel,75):7.4f}  "
      f"{np.percentile(d_novel,95):7.4f}")

# Nearest prototype for each novel class
print('\n--- Nearest prototype for novel classes (held-out) ---')
le_train_classes = [le_full.classes_[i] for i in le_train.classes_]
held_global_idx  = te_bal_idx[held_idx]
for dc in DROPPED_CLASSES:
    dc_id = le_full.transform([dc])[0]
    mask  = y_te[held_global_idx] == dc_id
    if not mask.any():
        continue
    _, preds_dc, dists_dc = detector.detect(X_held_t[mask])
    counts = [(le_train_classes[p], int((preds_dc == p).sum())) for p in range(num_train_classes)]
    counts.sort(key=lambda x: -x[1])
    n = int(mask.sum())
    print(f"  '{dc}' (n={n:,})  mean-dist={dists_dc.mean():.4f}")
    for name, cnt in counts[:3]:
        print(f'      -> {name}: {cnt:,} ({100*cnt/n:.1f}%)')

## 8. Summary

In [ ]:
print('=' * 65)
print(f"{'Method':<22} {'Precision':>9} {'Recall':>7} {'F1':>7} {'Threshold':>10}")
print('=' * 65)
print(f"{'ContrastiveNCM':<22} {p_c:>9.3f} {r_c:>7.3f} {f_c:>7.3f} {thresh_c:>10.4f}")
print(f"{'Plain AE NCM':<22} {p_p:>9.3f} {r_p:>7.3f} {f_p:>7.3f} {thresh_p:>10.4f}")
print('=' * 65)
print(f'\nDataset        : CICIDS2017 (Engelen corrected)')
print(f'Dropped classes: {DROPPED_CLASSES}')
print(f'Threshold      : val-optimal sweep [0.01,0.99] -> evaluated on disjoint held-out')
print(f'Note           : paper uses IDS2018; direct numeric comparison is approximate.')